In [ ]:
import sys
import os
import time
sys.path.append(os.path.abspath('..')) 
from drone_env.drone_sim import RoomDroneEnv
from stable_baselines3 import PPO
import pybullet as p

print("GUI simülasyonu Canlı Yayın için başlatılıyor...")
# Sadece izlemek için GUI açıyoruz
env = RoomDroneEnv(gui=True)

# ARKA PLANDA EĞİTİLEN VE SÜREKLİ GÜNCELLENEN MODELİ YÜKLÜYORUZ
model_path = os.path.abspath(os.path.join('..', 'models', 'best_model'))
print(f"Eğitilmiş model yükleniyor: {model_path}.zip")

# Modeli sadece "okuma" modunda yüklüyoruz, eğitime zerre zararı yok!
model = PPO.load(model_path, env=env)

obs, info = env.reset()
p.resetDebugVisualizerCamera(cameraDistance=6.0, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0, 0, 1])

print("Canlı Yayın Başladı! Çarpıya basana kadar devam edecek...")

while True:
    # ARTIK RASTGELE VEYA SABİT BİR KOMUT YOK! 
    # Dronun beynine o anki gözlemleri (obs) veriyoruz, o da ne yapacağına kendi karar veriyor!
    action, _states = model.predict(obs, deterministic=True)
    
    obs, reward, terminated, truncated, info = env.step(action) 
    time.sleep(1./240.) # Gerçek zamanlı akış
    
    if terminated or truncated:
        print("Bölüm Bitti! Yeni oda oluşturuluyor...")
        time.sleep(2) # Çarptıktan sonra 2 saniye bekle
        obs, info = env.reset()